# v5 跨会话标签风险域预检

冻结后只读 2018–2020 已保存的 v5 整股配对意向、逐层点时父本与 Qlib 双 ETF 分钟收盘。检验 +0.10 名义 beta 动作与旧父本上限的重叠；不读取未来动作价值、2021/2022 结果或生成新标签。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/"research/stage1_v5_path_label_domain_preflight_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
for key in ("checkpoint_code","dated_open_action_code"):
 assert sha(root/plan[key])==plan[key+"_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
for role in ("label_source","parent"):
 s=plan[role];source=root/s["card"]
 assert sha(source/"summary.json")==s["summary_sha256"] and sha(source/"verification_manifest.json")==s["manifest_sha256"]
 assert json.loads((source/"nas_binding.json").read_text())["evidence"]["sha256"]==s["nas_archive_sha256"]
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
def read(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s["card"],expected_recorder_id=s["recorder_id"],artifacts=[name],expected_manifest_sha256=s["manifest_sha256"])
 return v[name]
labels=read("label_source","analysis/labels")
book=read("parent","analysis/native/dated_f696_0")
assert len(labels)==384660 and len(book["independent_replay"])==1056
# Do not access label value columns. Funded flags are retrospective support, never live input.
intent=labels.loc[pd.to_datetime(labels.trade_date).dt.year.isin(plan["discovery_years"]) & labels.soxl_funded & labels.soxx_funded,
 ["trade_date","bar_end_at","decision_nav"]].copy()
assert len(intent)>0
print('verified pair intents',len(intent),'days',intent.trade_date.nunique())

[21632:MainThread](2026-10-30 13:26:21,573) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[21632:MainThread](2026-10-30 13:26:21,941) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[21632:MainThread](2026-10-30 13:26:21,942) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


verified pair intents 196841 days 547


In [2]:
bars=D.features(["SOXL","SOXX"],["$close","$price_valid"],start_time="2018-10-15 09:30:00",end_time="2020-12-31 16:00:00",freq="1min")
left=bars.xs("SOXL",level="instrument")[["$close","$price_valid"]].rename(columns={"$close":"soxl_close","$price_valid":"soxl_valid"})
right=bars.xs("SOXX",level="instrument")[["$close","$price_valid"]].rename(columns={"$close":"soxx_close","$price_valid":"soxx_valid"})
marks=left.join(right,how="inner").reindex(pd.to_datetime(intent.bar_end_at))
assert marks[["soxl_close","soxx_close"]].notna().all().all() and marks[["soxl_valid","soxx_valid"]].eq(1).all().all()
path=book["independent_replay"].set_index("trade_date")
q=[];cash=[];cap=[];recovery=[]
for day in pd.to_datetime(intent.trade_date):
 pos=book["native_positions"][day+pd.Timedelta(hours=9,minutes=47)]
 q.append([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ("SOXL","SOXX")])
 cash.append(float(pos.get_cash()))
 cap.append(float(path.loc[day,"current_beta_cap"]))
 recovery.append(bool(path.loc[day,"recovery_action"]))
assert not any(recovery)
q=np.asarray(q,float);cash=np.asarray(cash,float);cap=np.asarray(cap,float)
price=marks[["soxl_close","soxx_close"]].to_numpy(float);nav=(q*price).sum(axis=1)+cash
err=np.abs(nav-intent.decision_nav.to_numpy(float));assert err.max()<1e-6
w=(q*price)/nav[:,None];beta=3*w[:,0]+w[:,1]
proposed=beta+plan["action_nominal_beta_increment"]
already=beta>cap+1e-10;cross=~already & (proposed>cap+1e-10);beyond=proposed>cap+1e-10
result=pd.DataFrame({"trade_date":pd.to_datetime(intent.trade_date).to_numpy(),"year":pd.to_datetime(intent.trade_date).dt.year.to_numpy(),"already_over_old_cap":already,"newly_cross_old_cap":cross,"proposed_over_old_cap":beyond,"old_cap_headroom":cap-beta})
annual=[]
for year,g in result.groupby("year"):
 annual.append({"year":int(year),"paired_intent_rows":int(len(g)),"paired_days":int(g.trade_date.nunique()),"proposed_over_old_cap_rows":int(g.proposed_over_old_cap.sum()),"proposed_over_old_cap_days":int(g.loc[g.proposed_over_old_cap,"trade_date"].nunique()),"already_over_old_cap_rows":int(g.already_over_old_cap.sum()),"newly_cross_old_cap_rows":int(g.newly_cross_old_cap.sum()),"median_old_cap_headroom":float(g.old_cap_headroom.median())})
audit={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"paired_discovery_rows":len(result),"paired_discovery_days":int(result.trade_date.nunique()),"maximum_decision_nav_error_usd":float(err.max()),"proposed_over_old_cap_rows":int(beyond.sum()),"proposed_over_old_cap_days":int(result.loc[beyond,"trade_date"].nunique()),"already_over_old_cap_rows":int(already.sum()),"newly_cross_old_cap_rows":int(cross.sum()),"annual":annual,"checkpoint_subsequent_policy":"dated_b472_old_risk_architecture","new_fits":0,"new_accounts":0,"future_value_columns_accessed":False,"reserved_years_untouched":plan["reserved_years"],"decision":"old_cap_checkpoint_mixed_domain_for_v5_cap_free_path_labels" if beyond.any() else "old_cap_mismatch_not_observed_in_discovery"}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(audit,ensure_ascii=False,indent=2))

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_21632/2119430478.py:9: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pos=book["native_positions"][day+pd.Timedelta(hours=9,minutes=47)]


{
  "experiment": "stage1_v5_path_label_domain_preflight_20261030",
  "plan_sha256": "43546f023f9cddb7f67576acbef1ac273754b91c1f8a4b731763c5dcf053a379",
  "paired_discovery_rows": 196841,
  "paired_discovery_days": 547,
  "maximum_decision_nav_error_usd": 0.0,
  "proposed_over_old_cap_rows": 195717,
  "proposed_over_old_cap_days": 544,
  "already_over_old_cap_rows": 94842,
  "newly_cross_old_cap_rows": 100875,
  "annual": [
    {
      "year": 2018,
      "paired_intent_rows": 18114,
      "paired_days": 53,
      "proposed_over_old_cap_rows": 18114,
      "proposed_over_old_cap_days": 53,
      "already_over_old_cap_rows": 8187,
      "newly_cross_old_cap_rows": 9927,
      "median_old_cap_headroom": 0.0007814871070475071
    },
    {
      "year": 2019,
      "paired_intent_rows": 92051,
      "paired_days": 252,
      "proposed_over_old_cap_rows": 91681,
      "proposed_over_old_cap_days": 251,
      "already_over_old_cap_rows": 40971,
      "newly_cross_old_cap_rows": 50710,
      

## 中文结论

**SOXX 判定：0 个候选账户，合同 v2 的 0/2/5bp 门均未检验。** 只读 v5 已保存整股配对意向与逐层点时父本、Qlib 分钟价格；2018–2020 共 196841 行、547 日，事件时刻 NAV 与旧标签最大误差 0 美元。+0.10 名义 beta 后 195717 行（99.429%）、544 日越过旧父本上限；其中 94842 行动作前已因分钟行情越线，100875 行由本动作新越线。2018/2019/2020 越线分别 18114/91681/85922 行。旧上限仅作**跨架构错配诊断**，不是新无上限设计的容量门。

旧 checkpoint 的事件入口只检查三资产非负且和为1，后续 `resume_episode` 仍逐日调用旧 b472 的开盘和恢复风险投影。它可证明旧父本机械身份，不是无旧上限、训练/推理同域的跨会话行动教师。若直接把该续跑值用作 v5 cap-free 标签，动作在事件日越过旧上限、次日却回到旧控制器，会混合两套风险规则。当前因而停止批量路径标签和拟合；先冻结新分支的连续三资产风险域、后续动作规则、整股费用、期限和同状态对照，再另做无事件/事件机械身份门。2021/2022 价值列未参与计算，配对整股标志是未来填单条件，只用于本只读诊断，不可作为实时过滤器。